# Optimize Query Performance
Mesures effect of several diffrent optimizations of the queries.



## 1. Configure Spark

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, ensure_runtime, project_root

ensure_runtime()

from src.lake import GOLD, SILVER, read_delta, show_delta, write_gold

spark = create_spark("integration-gold")
ROOT = project_root()

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b8d2162b-59cd-4366-ae8c-d9ca35c119bf;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 100ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs


## 2. Load silver tables

In [2]:
trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

## 3. Benchmark and Verification 

In [3]:
import time

def benchmark_and_verify(df_baseline, df_optimized, name="Optimization Test"):
    print(f"=== {name} ===")
    
    print("\n--- Baseline Plan ---")
    df_baseline.explain("formatted")
    print("\n--- Optimized Plan ---")
    df_optimized.explain("formatted")
    
    # Measure Baseline
    t0 = time.time()
    base_count = df_baseline.count()
    base_time = time.time() - t0
    
    # Measure Optimized
    t0 = time.time()
    opt_count = df_optimized.count()
    opt_time = time.time() - t0
    
    assert base_count == opt_count, f"Mismatch! Baseline: {base_count}, Optimized: {opt_count}"
    
    if base_count < 10000:
        assert df_baseline.orderBy(df_baseline.columns).collect() == df_optimized.orderBy(df_optimized.columns).collect()
    
    speedup = ((base_time - opt_time) / base_time) * 100 if base_time > 0 else 0
    print(f"Row Count:      {base_count:,}")
    print(f"Baseline Time:  {base_time:.2f}s")
    print(f"Optimized Time: {opt_time:.2f}s")
    print(f"Speedup:        {speedup:.1f}%\n")

## 4. Caching Results

In [4]:
import time
from pyspark.sql.functions import broadcast, avg

weather_hourly = weather.groupBy("observation_date", "observation_hour") \
                        .agg(avg("temperature_c").alias("temperature_c"))

intermediate_df = trips.filter("fare_amount > 0 AND trip_distance > 0") \
                       .select("pickup_date", "pickup_hour", "fare_amount", "trip_distance") \
                       .join(
                           broadcast(weather_hourly),
                           (trips.pickup_date == weather_hourly.observation_date) & 
                           (trips.pickup_hour == weather_hourly.observation_hour),
                           "inner"
                       )

# Baseline: uncached
t0 = time.time()
count1_base = intermediate_df.groupBy("pickup_hour").avg("fare_amount").count()
count2_base = intermediate_df.groupBy("temperature_c").avg("trip_distance").count()
base_time = time.time() - t0

# Optimized: cached
t0 = time.time()
cached_df = intermediate_df.cache()
cached_df.count()
count1_opt = cached_df.groupBy("pickup_hour").avg("fare_amount").count()
count2_opt = cached_df.groupBy("temperature_c").avg("trip_distance").count()
opt_time = time.time() - t0

print("\n--- Cached Plan (showing InMemoryTableScan) ---")
cached_df.explain("formatted")

cached_df.unpersist()

print(f"=== Cache Performance ===")
print(f"Baseline Time (Uncached, 2 queries): {base_time:.2f}s")
print(f"Optimized Time (Cached, 2 queries):  {opt_time:.2f}s")
print(f"Speedup: {((base_time - opt_time)/base_time)*100:.1f}%")

assert count1_base == count1_opt and count2_base == count2_opt, "Caching Result Mismatch!"


26/09/24 16:23:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



--- Cached Plan (showing InMemoryTableScan) ---
== Physical Plan ==
AdaptiveSparkPlan (28)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- AdaptiveSparkPlan (27)
               +- == Final Plan ==
                  * BroadcastHashJoin Inner BuildRight (17)
                  :- * Project (6)
                  :  +- * Filter (5)
                  :     +- * ColumnarToRow (4)
                  :        +- Scan parquet  (3)
                  +- BroadcastQueryStage (16), Statistics(sizeInBytes=32.5 MiB, rowCount=8.78E+3)
                     +- BroadcastExchange (15)
                        +- * HashAggregate (14)
                           +- AQEShuffleRead (13)
                              +- ShuffleQueryStage (12), Statistics(sizeInBytes=343.1 KiB, rowCount=8.78E+3)
                                 +- Exchange (11)
                                    +- * HashAggregate (10)
                                       +- * Filter (9)
                                    

## 5. Partition Pruning

In [5]:
# Baseline: filtering on timestamp expression
df_base = trips.filter("date(pickup_datetime) = '2024-01-01'")

# Optimized: explicitly filtering on the partition column 'pickup_date'
df_opt = trips.filter("pickup_date = '2024-01-01' AND date(pickup_datetime) = '2024-01-01'")

benchmark_and_verify(df_base, df_opt, "Partition Pruning Performance")


=== Partition Pruning Performance ===

--- Baseline Plan ---
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [14]: [taxi_type#48, vendor_id#49, pickup_datetime#50, dropoff_datetime#51, passenger_count#52, trip_distance#53, pickup_location_id#54, dropoff_location_id#55, fare_amount#56, tip_amount#57, tolls_amount#58, total_amount#59, pickup_hour#61, pickup_date#60]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/silver/taxi_trips]
PushedFilters: [IsNotNull(pickup_datetime), GreaterThanOrEqual(pickup_datetime,2024-01-01 01:00:00.0), LessThan(pickup_datetime,2024-01-02 01:00:00.0)]
ReadSchema: struct<taxi_type:string,vendor_id:int,pickup_datetime:timestamp,dropoff_datetime:timestamp,passenger_count:int,trip_distance:double,pickup_location_id:int,dropoff_location_id:int,fare_amount:double,tip_amount:double,tolls_amount:double,total_am

## 6. Optimize Broadcast Joins

In [6]:
from pyspark.sql.functions import broadcast

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

df_base = trips.join(zones, trips.pickup_location_id == zones.location_id)
df_opt = trips.join(broadcast(zones), trips.pickup_location_id == zones.location_id)

benchmark_and_verify(df_base, df_opt, "Broadcast Join Performance")

=== Broadcast Join Performance ===

--- Baseline Plan ---
== Physical Plan ==
AdaptiveSparkPlan (11)
+- SortMergeJoin Inner (10)
   :- Sort (5)
   :  +- Exchange (4)
   :     +- Project (3)
   :        +- Filter (2)
   :           +- Scan parquet  (1)
   +- Sort (9)
      +- Exchange (8)
         +- Filter (7)
            +- Scan parquet  (6)


(1) Scan parquet 
Output [14]: [taxi_type#48, vendor_id#49, pickup_datetime#50, dropoff_datetime#51, passenger_count#52, trip_distance#53, pickup_location_id#54, dropoff_location_id#55, fare_amount#56, tip_amount#57, tolls_amount#58, total_amount#59, pickup_hour#61, pickup_date#60]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/silver/taxi_trips]
PushedFilters: [IsNotNull(pickup_location_id)]
ReadSchema: struct<taxi_type:string,vendor_id:int,pickup_datetime:timestamp,dropoff_datetime:timestamp,passenger_count:int,trip_distance:double,pickup_location_id:int,dropoff_locat

Row Count:      9,417,383
Baseline Time:  2.69s
Optimized Time: 1.06s
Speedup:        60.8%



## 7. Adaptive Query Execution (AQE)

In [7]:
query_df = trips.groupBy("pickup_location_id", "pickup_hour") \
                .agg({"fare_amount": "avg", "trip_distance": "sum"})


# Baseline: AQE disabled
spark.conf.set("spark.sql.adaptive.enabled", "false")
print("--- AQE Disabled Plan ---")
query_df.explain("formatted")

t0 = time.time()
count_disabled = query_df.count()
time_disabled = time.time() - t0

# Optimized: AQE enabled
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

t0 = time.time()
count_enabled = query_df.count()
time_enabled = time.time() - t0

print("\n--- AQE Enabled Plan ---")
query_df.explain("formatted")

assert count_disabled == count_enabled, "AQE Result Mismatch!"

print(f"\n=== AQE Performance ===")
print(f"AQE Disabled Time: {time_disabled:.2f}s")
print(f"AQE Enabled Time:  {time_enabled:.2f}s")
print(f"Speedup:           {((time_disabled - time_enabled)/time_disabled)*100:.1f}%\n")

--- AQE Disabled Plan ---
== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- * Project (3)
         +- * ColumnarToRow (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61, pickup_date#60]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/silver/taxi_trips]
ReadSchema: struct<trip_distance:double,pickup_location_id:int,fare_amount:double,pickup_hour:int>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61, pickup_date#60]

(3) Project [codegen id : 1]
Output [4]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61]
Input [5]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61, pickup_date#60]

(4) HashAggregate [codegen id : 1]
Input [4]: [trip_distance#53, pickup_location_id#5


--- AQE Enabled Plan ---
== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- * Project (3)
         +- * ColumnarToRow (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61, pickup_date#60]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/silver/taxi_trips]
ReadSchema: struct<trip_distance:double,pickup_location_id:int,fare_amount:double,pickup_hour:int>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61, pickup_date#60]

(3) Project [codegen id : 1]
Output [4]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61]
Input [5]: [trip_distance#53, pickup_location_id#54, fare_amount#56, pickup_hour#61, pickup_date#60]

(4) HashAggregate [codegen id : 1]
Input [4]: [trip_distance#53, pickup_location_id#5